In [ ]:
import math
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import FancyBboxPatch
import geopandas as gpd
from pathlib import Path
import seaborn as sns

In [ ]:
sns.set_theme(style="white")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#4a4a4a",
    "axes.linewidth": 0.6,
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
    "figure.titlesize": 16,
    "figure.titleweight": "bold",
    "figure.dpi": 150
})

OUTPUT_DIR = Path("./figures")
OUTPUT_DIR.mkdir(exist_ok=True)

LAYERS_DIR = Path("./data/silver/layers")
BOUNDARY_PATH = LAYERS_DIR / "boundary.shp"

SEASONS = ["spring", "summer", "fall"]
DENSITY_METHODS = ["convolution", "kde"]
CLASSIFY_METHODS = ["percentile", "jenks", "gmm"]
CLASS_LABELS = ["Low", "Medium", "High", "Very High"]

BOUNDARY_COLOR = "#2b2b2b"
BOUNDARY_LW = 1.1
MISSING_COLOR = "#c0392b"

_boundary_cache = None

In [ ]:
def get_boundary() -> gpd.GeoDataFrame:
    """Load and cache the Essex boundary — avoids re-reading the shapefile per plot."""
    global _boundary_cache
    if _boundary_cache is None:
        _boundary_cache = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:27700")
    return _boundary_cache

In [ ]:
def _style_map_axis(ax):
    """Strip chart-junk: no tick marks/labels, thin neat border, clean look for small maps."""
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#4a4a4a")
        spine.set_linewidth(0.6)


def _render_missing(ax, filename, title):
    """Softer, more legible 'missing file' placeholder than raw red text."""
    ax.set_facecolor("#f7f2f2")
    box = FancyBboxPatch(
        (0.08, 0.35), 0.84, 0.3, transform=ax.transAxes,
        boxstyle="round,pad=0.02", linewidth=1, edgecolor=MISSING_COLOR,
        facecolor="white",
    )
    ax.add_patch(box)
    ax.text(0.5, 0.5, f"missing:\n{filename}", ha="center", va="center",
             transform=ax.transAxes, fontsize=9, color=MISSING_COLOR, fontweight="bold")
    ax.set_title(title, fontsize=11, fontweight="bold", color="#666666")
    _style_map_axis(ax)


def _render_raster(ax, path, title, cmap, norm=None, vmin=None, vmax=None, unit=""):
    """Render one raster onto an axis with boundary overlay and clean styling. Returns the image handle."""
    with rasterio.open(path) as src:
        data = src.read(1)
        bounds = src.bounds

    im = ax.imshow(
        data, extent=[bounds.left, bounds.right, bounds.bottom, bounds.top],
        cmap=cmap, norm=norm, vmin=vmin, vmax=vmax,
        interpolation="nearest", origin="upper",
    )
    get_boundary().boundary.plot(ax=ax, color=BOUNDARY_COLOR, linewidth=BOUNDARY_LW)
    ax.set_title(title, fontsize=11, fontweight="bold")
    _style_map_axis(ax)
    return im


def _save_and_show(fig, output_path, save, show):
    if save:
        fig.savefig(output_path, dpi=170, bbox_inches="tight", facecolor="white")
        print(f"  Saved: {output_path.name}")
    if show:
        plt.show()
    else:
        plt.close(fig)

# Grouped

In [ ]:
def plot_raster_group(var_list, group_title, filename_prefix, ncols=3, save=True, show=True):
    """
    var_list: list of dicts {filename, title, cmap, unit (optional)}.
    Each variable gets its own colorbar since units generally differ across
    a "type" group (e.g. climate mixes °C, mm, m/s, %).
    """
    n_vars = len(var_list)
    cols = min(ncols, n_vars)
    rows = math.ceil(n_vars / cols)

    fig = plt.figure(figsize=(5.2 * cols, 4.6 * rows), constrained_layout=True)
    gs = GridSpec(rows, cols, figure=fig)

    missing = []
    for idx, var in enumerate(var_list):
        ax = fig.add_subplot(gs[idx])
        path = LAYERS_DIR / var["filename"]

        if not path.exists():
            missing.append(var["filename"])
            _render_missing(ax, var["filename"], var["title"])
            continue

        im = _render_raster(ax, path, var["title"], var["cmap"])
        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
        cbar.set_label(var.get("unit", ""), fontsize=8)
        cbar.ax.tick_params(labelsize=7)

    for idx in range(n_vars, rows * cols):
        fig.add_subplot(gs[idx]).axis("off")

    fig.suptitle(group_title)

    if missing:
        print(f"  [!] {group_title}: {len(missing)} missing tif(s) -> {missing}")

    _save_and_show(fig, OUTPUT_DIR / f"{filename_prefix}.png", save, show)

In [ ]:
def plot_density_comparison(season: str, density_methods=DENSITY_METHODS, save=True, show=True):
    """
    Fire density surfaces side by side (one panel per density_method), sharing
    a single color scale so magnitudes are directly comparable — convolution
    and KDE densities live on different numeric scales, so vmin/vmax here are
    computed per-panel's own p99 rather than forced equal; the goal is shape
    comparison, not literal magnitude equivalence.
    """
    n = len(density_methods)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5.2), constrained_layout=True)
    if n == 1:
        axes = [axes]

    missing = []
    for ax, method in zip(axes, density_methods):
        tag = method[:4]  # matches fire_density_conv_*.tif / fire_density_kde_*.tif
        filename = f"fire_density_{tag}_{season}.tif"
        path = LAYERS_DIR / filename

        if not path.exists():
            missing.append(filename)
            _render_missing(ax, filename, method)
            continue

        with rasterio.open(path) as src:
            data = src.read(1)
        vmax = np.nanpercentile(data, 99)  # robust to outlier spikes near fire clusters

        im = _render_raster(ax, path, method, cmap="inferno", vmin=0, vmax=vmax)
        cbar = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
        cbar.set_label("density (p99-clipped)", fontsize=8)
        cbar.ax.tick_params(labelsize=7)

    fig.suptitle(f"Fire Density — {season.capitalize()}: Convolution vs. KDE")

    if missing:
        print(f"  [!] density comparison ({season}): missing -> {missing}")

    _save_and_show(fig, OUTPUT_DIR / f"fire_density_comparison_{season}.png", save, show)

In [ ]:
def plot_classify_comparison(
        season: str, 
        density_method="convolution",
        classify_methods=CLASSIFY_METHODS, 
        save=True, 
        show=True
    ):
    """
    Risk-label maps for every classify_method on one season, sharing a single
    discrete 4-class colormap/legend so panels are directly comparable.
    Class pixel counts are annotated in each subtitle.
    """
    n = len(classify_methods)
    fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5.4), constrained_layout=True)
    if n == 1:
        axes = [axes]

    cmap = ListedColormap(sns.color_palette("YlOrRd", 4))
    norm = BoundaryNorm(boundaries=[-0.5, 0.5, 1.5, 2.5, 3.5], ncolors=4)

    missing = []
    last_im = None

    for ax, classify_method in zip(axes, classify_methods):
        filename = f"risk_labels_{density_method}_{classify_method}_{season}.tif"
        path = LAYERS_DIR / filename

        if not path.exists():
            missing.append(filename)
            _render_missing(ax, filename, classify_method)
            continue

        with rasterio.open(path) as src:
            data = src.read(1)

        last_im = _render_raster(ax, path, "", cmap=cmap, norm=norm)
        valid = data[~np.isnan(data)]
        counts = {CLASS_LABELS[c]: int((valid == c).sum()) for c in range(4)}
        count_str = " | ".join(f"{k}: {v:,}" for k, v in counts.items())
        ax.set_title(f"{classify_method}", fontsize=12, fontweight="bold")
        ax.text(0.5, -0.06, count_str, transform=ax.transAxes,
                 ha="center", va="top", fontsize=7.5, color="#444444")

    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes, fraction=0.03, pad=0.02, ticks=[0, 1, 2, 3])
        cbar.ax.set_yticklabels(CLASS_LABELS)
        cbar.set_label("Susceptibility class", fontsize=9)

    fig.suptitle(f"Label Classification — {season.capitalize()} ({density_method})")

    if missing:
        print(f"  [!] classify comparison ({season}): missing -> {missing}")

    _save_and_show(fig, OUTPUT_DIR / f"risk_labels_comparison_{season}.png", save, show)

In [ ]:
def topo_vars():
    return [
        {"filename": "topo_elevation.tif", "title": "Elevation", "cmap": "terrain", "unit": "m"},
        {"filename": "topo_slope.tif", "title": "Slope", "cmap": "YlOrBr", "unit": "degrees"},
        {"filename": "topo_aspect.tif", "title": "Aspect", "cmap": "twilight", "unit": "degrees"},
    ]


def proximity_vars():
    """Static proximity features only — d_fires lives in the fire group, not here."""
    return [
        {"filename": "dist_activity.tif", "title": "Distance to Human Activity", "cmap": "magma_r", "unit": "km"},
        {"filename": "dist_rivers.tif", "title": "Distance to Rivers", "cmap": "Blues_r", "unit": "km"},
        {"filename": "dist_roads.tif", "title": "Distance to Roads", "cmap": "cividis_r", "unit": "km"},
    ]


def climate_vars(season: str):
    return [
        {"filename": f"meteo_tas_{season}.tif", "title": "Mean Temperature", "cmap": "RdYlBu_r", "unit": "°C"},
        {"filename": f"meteo_tasmax_{season}.tif", "title": "Max Temperature", "cmap": "Reds", "unit": "°C"},
        {"filename": f"meteo_tasmin_{season}.tif", "title": "Min Temperature", "cmap": "Blues", "unit": "°C"},
        {"filename": f"meteo_rainfall_{season}.tif", "title": "Precipitation", "cmap": "Blues", "unit": "mm"},
        {"filename": f"meteo_hurs_{season}.tif", "title": "Relative Humidity", "cmap": "Greens", "unit": "%"},
        {"filename": f"meteo_sfcWind_{season}.tif", "title": "Wind Speed", "cmap": "Purples", "unit": "m/s"},
    ]


def ndvi_vars(season: str):
    return [
        {"filename": f"ndvi_{season}.tif", "title": f"NDVI — {season.capitalize()}", "cmap": "YlGn", "unit": "index"},
    ]


def fire_proximity_vars(season: str):
    return [
        {"filename": f"dist_fires_{season}.tif", "title": f"Distance to Fires — {season.capitalize()}",
         "cmap": "magma_r", "unit": "km"},
    ]

In [ ]:
print("=== Topography (static) ===")
plot_raster_group(topo_vars(), "Topographical Variables", "01_topo_maps")

print("\n=== Proximity (static) ===")
plot_raster_group(proximity_vars(), "Proximity Features (Static)", "02_proximity_maps")

print("\n=== Climate (seasonal) ===")
for season in SEASONS:
    plot_raster_group(climate_vars(season), f"Climate Variables — {season.capitalize()}",
                      f"03_climate_maps_{season}")

print("\n=== NDVI (seasonal) ===")
for season in SEASONS:
    plot_raster_group(ndvi_vars(season), f"NDVI — {season.capitalize()}",
                      f"04_ndvi_map_{season}", ncols=1)

print("\n=== Fire (proximity + density + labels, seasonal) ===")
for season in SEASONS:
    print(f"  -- {season} --")
    plot_raster_group(fire_proximity_vars(season), f"Distance to Fires — {season.capitalize()}",
                      f"05a_fire_proximity_{season}", ncols=1)
    plot_density_comparison(season)
    plot_classify_comparison(season)

# Individual

In [ ]:
def save_individual_plots(var_list, output_subfolder="individual_maps"):
    save_dir = OUTPUT_DIR / output_subfolder
    save_dir.mkdir(parents=True, exist_ok=True)

    for var in var_list:
        path = LAYERS_DIR / var["filename"]
        if not path.exists():
            print(f"  Skipping {var['filename']}: not found.")
            continue

        fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
        im = _render_raster(ax, path, var["title"], var["cmap"])
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
        cbar.set_label(var.get("unit", ""), fontsize=8)

        out_name = var["filename"].replace(".tif", ".png").replace(".gpkg", ".png")
        output_path = save_dir / out_name
        fig.savefig(output_path, dpi=170, bbox_inches="tight", facecolor="white")
        print(f"  Saved: {output_path.name}")
        plt.close(fig)

In [ ]:

print("\nSaving individual maps...")
save_individual_plots(topo_vars(), output_subfolder="topography")
save_individual_plots(proximity_vars(), output_subfolder="proximity")
for season in SEASONS:
    save_individual_plots(climate_vars(season), output_subfolder=f"climate/{season}")
    save_individual_plots(ndvi_vars(season), output_subfolder=f"ndvi/{season}")
    save_individual_plots(fire_proximity_vars(season), output_subfolder=f"fire/{season}")

print("\nAll maps generated successfully.")